This section reads from the Delta raw table as a stream, parses the `body_str` JSON payload into explicit columns, keeps the raw `ingested_at` timestamp, and writes the result to `dev.bronze.ttc_alerts_bronze`.

In [0]:
from pyspark.sql import functions as F, types as T

source_table = "dev.raw.weather_raw"
target_table_actual = "dev.bronze.weather_bronze"
target_table_forecast = "dev.bronze.weather_forecast_bronze"
checkpoint_path = "/Volumes/dev/bronze/checkpoints/weather_bronze"

weather_schema = T.StructType([
    T.StructField("name", T.StringType(), True),
    T.StructField("region", T.StringType(), True),
    T.StructField("country", T.StringType(), True),
    T.StructField("lat", T.DoubleType(), True),
    T.StructField("lon", T.DoubleType(), True),
    T.StructField("localtime", T.StringType(), True),
    T.StructField("temp_c", T.DoubleType(), True),
    T.StructField("is_day", T.IntegerType(), True),
    T.StructField("condition_text", T.StringType(), True),
    T.StructField("condition_icon", T.StringType(), True),
    T.StructField("wind_kph", T.DoubleType(), True),
    T.StructField("wind_degree", T.IntegerType(), True),
    T.StructField("wind_dir", T.StringType(), True),
    T.StructField("pressure_in", T.DoubleType(), True),
    T.StructField("precip_in", T.DoubleType(), True),
    T.StructField("humidity", T.IntegerType(), True),
    T.StructField("cloud", T.IntegerType(), True),
    T.StructField("feelslike_c", T.DoubleType(), True),
    T.StructField("uv", T.DoubleType(), True),
    T.StructField("air_quality", T.StructType([
        T.StructField("co", T.DoubleType(), True),
        T.StructField("no2", T.DoubleType(), True),
        T.StructField("o3", T.DoubleType(), True),
        T.StructField("so2", T.DoubleType(), True),
        T.StructField("pm2_5", T.DoubleType(), True),
        T.StructField("pm10", T.DoubleType(), True),
        T.StructField("us-epa-index", T.IntegerType(), True),
        T.StructField("gb-defra-index", T.IntegerType(), True)
    ]), True),
    T.StructField("alerts", T.ArrayType(T.StringType()), True),
    T.StructField("forecast", T.ArrayType(
        T.StructType([
            T.StructField("date", T.StringType(), True),
            T.StructField("maxtemp_c", T.DoubleType(), True),
            T.StructField("mintemp_c", T.DoubleType(), True),
            T.StructField("condition", T.StringType(), True)
        ])
    ), True)
])

In [0]:
raw_stream = spark.readStream.table(source_table)

bronze_stream_weather = (
    raw_stream
    .select(
        F.from_json(F.col("body_str"), weather_schema).alias("payload"),
        F.col("ingested_at"),
    )
    .select(
        F.col("payload.name").alias("city"),
        F.col("payload.region").alias("region"),
        F.col("payload.country").alias("country"),
        F.col("payload.lat").alias("latitude"),
        F.col("payload.lon").alias("longitude"),
        F.to_timestamp(F.col("payload.localtime"), "yyyy-MM-dd HH:mm").alias("localtime"),
        F.col("payload.temp_c").alias("temperature_c"),
        F.col("payload.is_day").alias("is_day"),
        F.col("payload.condition_text").alias("condition_text"),
        F.col("payload.condition_icon").alias("condition_icon"),
        F.col("payload.wind_kph").alias("wind_kph"),
        F.col("payload.wind_degree").alias("wind_degree"),
        F.col("payload.wind_dir").alias("wind_direction"),
        F.col("payload.pressure_in").alias("pressure_in"),
        F.col("payload.precip_in").alias("precipitation_in"),
        F.col("payload.humidity").alias("humidity"),
        F.col("payload.cloud").alias("cloud"),
        F.col("payload.feelslike_c").alias("feelslike_c"),
        F.col("payload.uv").alias("uv_index"),
        F.col("payload.air_quality.co").alias("air_quality_co"),
        F.col("payload.air_quality.no2").alias("air_quality_no2"),
        F.col("payload.air_quality.o3").alias("air_quality_o3"),
        F.col("payload.air_quality.so2").alias("air_quality_so2"),
        F.col("payload.air_quality.pm2_5").alias("air_quality_pm2_5"),
        F.col("payload.air_quality.pm10").alias("air_quality_pm10"),
        F.col("payload.air_quality.us-epa-index").alias("air_quality_us_epa_index"),
        F.col("payload.air_quality.gb-defra-index").alias("air_quality_gb_defra_index"),
        F.col("ingested_at").alias("ingested_at")
    )
)

In [0]:
bronze_stream_weather_forecast = (
    raw_stream
    .select(
        F.from_json(F.col("body_str"), weather_schema).alias("payload"),
        F.col("ingested_at"),
    )
    .withColumn("forecast", F.explode_outer("payload.forecast"))
    .select(
        F.col("forecast.date").alias("date"),
        F.col("forecast.maxtemp_c").alias("maxtemp_c"),
        F.col("forecast.mintemp_c").alias("mintemp_c"),
        F.col("forecast.condition").alias("condition"),
        F.col("ingested_at")
    )
)

In [0]:
query_weather = (
    bronze_stream_weather.writeStream
    .format("delta")
    .outputMode("append")
    .trigger(availableNow=True)
    .option("checkpointLocation", checkpoint_path + "/actual")
    .toTable(target_table_actual)
)

query_forecast = (
    bronze_stream_weather_forecast.writeStream
    .format("delta")
    .outputMode("append")
    .trigger(availableNow=True)
    .option("checkpointLocation", checkpoint_path + "/forecast")
    .toTable(target_table_forecast)
)